# Week 4 — Training a DDPM (Reference Solution)

> **Mentor reference.** Keep private until after the Week 4 deadline. **This is the most important week.**

Combines everything so far into a working **Denoising Diffusion Probabilistic Model**. We add **timestep conditioning** to the Week 2 UNet, train it to predict the noise added at each step (DDPM Algorithm 1), and generate digits from pure noise via the reverse process (DDPM Algorithm 2).

**Runtime:** ~15–20 min on Colab GPU for recognizable digits. Use Runtime → Change runtime type → T4 GPU. CPU will be very slow — reduce `EPOCHS` and image count to smoke-test.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Data and scheduler (from Week 3)

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train = datasets.MNIST("./data", train=True, download=True, transform=tf)
loader = DataLoader(train, batch_size=128, shuffle=True, drop_last=True)

T = 1000
betas = torch.linspace(1e-4, 0.02, T).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
sqrt_acp = torch.sqrt(alphas_cumprod)
sqrt_one_minus_acp = torch.sqrt(1.0 - alphas_cumprod)

def q_sample(x0, t, noise):
    a = sqrt_acp[t].view(-1, 1, 1, 1)
    b = sqrt_one_minus_acp[t].view(-1, 1, 1, 1)
    return a * x0 + b * noise

## 2. Timestep embedding

The network must know *which* noise level it's denoising. We encode the integer timestep with a sinusoidal embedding (same idea as transformer positional encodings), then project it and inject it into every UNet block.

In [ ]:
def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

## 3. UNet with timestep conditioning

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.temb = nn.Linear(tdim, cout)
        self.norm1 = nn.GroupNorm(8, cout) if cout % 8 == 0 else nn.GroupNorm(1, cout)
        self.norm2 = nn.GroupNorm(8, cout) if cout % 8 == 0 else nn.GroupNorm(1, cout)
        self.act = nn.SiLU()
    def forward(self, x, t):
        h = self.act(self.norm1(self.conv1(x)))
        h = h + self.temb(t)[:, :, None, None]   # inject timestep
        h = self.act(self.norm2(self.conv2(h)))
        return h

class UNet(nn.Module):
    def __init__(self, ch=1, base=64, tdim=128):
        super().__init__()
        self.tdim = tdim
        self.tmlp = nn.Sequential(nn.Linear(tdim, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.d1 = Block(ch, base, tdim)
        self.d2 = Block(base, base * 2, tdim)
        self.pool = nn.MaxPool2d(2)
        self.mid = Block(base * 2, base * 2, tdim)
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.u2 = Block(base * 2 + base * 2, base, tdim)
        self.u1 = Block(base + base, base, tdim)
        self.out = nn.Conv2d(base, ch, 1)
    def forward(self, x, t):
        temb = self.tmlp(sinusoidal_embedding(t, self.tdim))
        s1 = self.d1(x, temb)
        s2 = self.d2(self.pool(s1), temb)
        m = self.mid(self.pool(s2), temb)
        u = self.up(m)
        # handle odd sizes (28 -> 14 -> 7 -> 14 needs care)
        if u.shape[-1] != s2.shape[-1]:
            u = F.interpolate(u, size=s2.shape[-2:])
        u = self.u2(torch.cat([u, s2], 1), temb)
        u = self.up(u)
        if u.shape[-1] != s1.shape[-1]:
            u = F.interpolate(u, size=s1.shape[-2:])
        u = self.u1(torch.cat([u, s1], 1), temb)
        return self.out(u)

model = UNet().to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")
# shape check
_t = torch.randn(2, 1, 28, 28, device=device)
assert model(_t, torch.tensor([0, 1], device=device)).shape == _t.shape
print("shape check passed")

## 4. Training loop (DDPM Algorithm 1)

For each batch: pick random timesteps, noise the images to that level, ask the model to predict the noise, minimize MSE between predicted and true noise.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=2e-4)
EPOCHS = 10            # raise to 20-30 for sharper samples
losses = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for x, _ in loader:
        x = x.to(device)
        t = torch.randint(0, T, (x.size(0),), device=device)
        noise = torch.randn_like(x)
        xt = q_sample(x, t, noise)
        pred = model(xt, t)
        loss = F.mse_loss(pred, noise)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item() * x.size(0)
    epoch_loss = running / len(train)
    losses.append(epoch_loss)
    print(f"epoch {epoch:2d} | noise-mse {epoch_loss:.4f}")

## 5. Sampling (DDPM Algorithm 2)

Start from pure noise and iteratively denoise, one timestep at a time, all the way back to `t=0`.

In [ ]:
@torch.no_grad()
def sample(model, n=16, size=28):
    model.eval()
    x = torch.randn(n, 1, size, size, device=device)
    for i in reversed(range(T)):
        t = torch.full((n,), i, dtype=torch.long, device=device)
        eps = model(x, t)
        alpha = alphas[i]; acp = alphas_cumprod[i]; beta = betas[i]
        mean = (1 / torch.sqrt(alpha)) * (x - (beta / torch.sqrt(1 - acp)) * eps)
        if i > 0:
            x = mean + torch.sqrt(beta) * torch.randn_like(x)
        else:
            x = mean
    return x

samples = sample(model, n=16)
print("has_nan:", torch.isnan(samples).any().item())
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for ax, im in zip(axes.flatten(), samples):
    ax.imshow(((im[0].cpu().clamp(-1, 1) + 1) / 2), cmap="gray"); ax.axis("off")
plt.suptitle("Generated from pure noise"); plt.tight_layout(); plt.show()

## 6. Loss curve and checkpoint

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("noise MSE"); plt.title("DDPM training loss")
plt.show()

torch.save({"model_state": model.state_dict(),
            "T": T, "losses": losses}, "ddpm_mnist.pt")
print("checkpoint saved: ddpm_mnist.pt")

## Self-check answers (for mentor)

1. **Algorithm 1 (training) in words:** sample `x0`, sample `t` uniformly, sample noise ε, form `x_t` with `q_sample`, predict ε̂ = model(x_t, t), minimize ‖ε − ε̂‖².
2. **Algorithm 2 (sampling) in words:** start `x_T ~ N(0,I)`; for `t` from `T-1` to `0`, predict noise, compute the posterior mean, add fresh noise except at the final step.
3. **Why predict noise instead of `x_{t-1}`?** Empirically easier to learn and gives a simpler MSE objective; the two are mathematically linked via the closed form.
4. **Timestep embedding injection:** sinusoidal encode `t`, MLP project, add as a per-channel bias inside each block.

## Common mentee mistakes (the ones that eat office hours)
- **No timestep embedding** → trains but generates noise.
- **Sign error in the sampling mean** → samples blow up to NaN.
- **Too few epochs** → declares it broken at epoch 3; MNIST needs ~10+ to look digit-like, 20–30 to look clean.
- **Forgetting `model.eval()` / `torch.no_grad()`** during sampling.